# Quantium Customer Analytics

## Project Overview

This project analyzes customer purchasing behaviour within the chips category using retail transaction and customer demographic data provided by Quantium.

The objective is to identify key customer segments, purchasing patterns, brand performance, and product preferences that can support data-driven business decisions.

This project was completed as part of the Quantium Data Analytics Virtual Experience Program.


## Business Objectives

The primary objectives of this analysis are:

- Assess the quality of the transaction and customer datasets.
- Clean and prepare the data for analysis.
- Identify high-value customer segments.
- Analyze purchasing behaviour across customer groups.
- Evaluate brand performance.
- Examine pack size preferences.
- Generate business insights and strategic recommendations.


## Dataset Description

This analysis uses two datasets provided by Quantium.

### Transaction Dataset
The transaction dataset contains information about individual chip purchases, including:

- Transaction date
- Store number
- Loyalty card number
- Product name
- Brand
- Pack size
- Quantity purchased
- Total sales

### Customer Dataset
The customer dataset contains demographic information for each loyalty card holder, including:

- Lifestage
- Premium customer classification

The two datasets are merged using the **LYLTY_CARD_NBR** field to create a single dataset for customer analytics.

# Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Plot style
plt.style.use("ggplot")

## Load the Datasets

The transaction and customer datasets are loaded into pandas DataFrames for data cleaning, integration, and analysis.

In [ ]:
transactions=pd.read_csv('QVI_transaction_data.csv') 
customers=pd.read_csv('QVI_purchase_behaviour.csv')

## Initial Data Inspection

Before cleaning the data, we inspect the datasets to understand their structure, identify missing values, and verify that the data has loaded correctly.

In [ ]:
transactions.head()


In [ ]:
customers.head()

# Display information about the transaction dataset

In [ ]:
transactions.info()

# Display information about the customer dataset

In [ ]:
customers.info()

# Display the shape of both datasets


In [ ]:
print("Transaction Dataset Shape:", transactions.shape)
print("Customer Dataset Shape:", customers.shape)

### Duplicate Record Assessment

Duplicate records can lead to biased results and incorrect business insights. Therefore, both datasets are checked for duplicate entries before further analysis.

In [ ]:
# Check for duplicate records
print("Duplicate rows in transaction dataset:", transactions.duplicated().sum())

print("Duplicate rows in customer dataset:", customers.duplicated().sum())

### Duplicate Record Verification

A duplicate record was detected in the transaction dataset. Before removing it, the duplicate records are inspected to confirm that they are exact duplicates.

In [ ]:
# Display duplicate rows
duplicate_rows = transactions[transactions.duplicated()]

duplicate_rows

# Display all occurrences of duplicate records (original + duplicate)

In [ ]:
transactions[transactions.duplicated(keep=False)]

In [ ]:
# Remove duplicate records
transactions = transactions.drop_duplicates()

# Verify duplicates have been removed
print("Remaining duplicate rows:", transactions.duplicated().sum())

### Missing Value Assessment

Missing values can affect the accuracy of analysis and statistical calculations. Therefore, both datasets are examined to identify any missing values before proceeding with further analysis.

In [ ]:
# Check missing values in the transaction dataset
transactions.isnull().sum()

In [ ]:
# Check missing values in the customer dataset
customers.isnull().sum()

### Missing Value Summary

No missing values were identified in either the transaction or customer datasets. Therefore, no additional data imputation or record removal was required before proceeding with the analysis.

### Data Type Assessment

The data types of each variable are reviewed to ensure they are appropriate for analysis. Correct data types are essential for accurate calculations, filtering, and visualization.

In [ ]:
transactions.dtypes

### Date Conversion

The **DATE** column is stored as an Excel serial number. It is converted to a standard datetime format to enable time-based analysis such as monthly sales trends and transaction timelines.

In [ ]:
# Convert Excel serial date to datetime format
transactions["DATE"] = pd.to_datetime(
    transactions["DATE"],
    origin="1899-12-30",
    unit="D"
)

# Verify the updated data type
transactions.dtypes

## Feature Engineering

Additional variables are created from the existing data to support customer and product-level analysis. These engineered features improve the ability to analyze brand performance and pack size preferences.

### Extracting Pack Size

Pack size is extracted from the product name using regular expressions. This allows analysis of customer preferences across different product sizes.

In [ ]:
# Extract pack size (grams) from the product name
transactions["PACK_SIZE"] = transactions["PROD_NAME"].str.extract(r"(\d+)").astype(int)

In [ ]:
transactions[["PROD_NAME", "PACK_SIZE"]].head()

### Extracting Brand Names

Brand names are extracted from the product description to enable brand-level sales analysis and comparison.

In [ ]:
# Extract brand names from the product description
transactions['BRAND'] = transactions['PROD_NAME'].str.split().str[0]

In [ ]:
transactions['BRAND'].nunique()

In [ ]:
transactions[['PROD_NAME', 'BRAND']].head()

### Standardizing Brand Names

Some brand names contain spelling variations or abbreviations that represent the same brand. These inconsistencies are corrected to ensure accurate brand-level analysis and reporting.

In [ ]:
# Mapping inconsistent brand names to standardized names
brand_mapping = {
    'Dorito': 'Doritos',
    'Infzns': 'Infuzions',
    'Smith': 'Smiths',
    'RRD': 'Red Rock Deli',
    'Red': 'Red Rock Deli',
    'NCC': 'Natural Chip Co',
    'Natural': 'Natural Chip Co',
    'Grain': 'Grain Waves',
    'GrnWves': 'Grain Waves',
    'Snbts': 'Sunbites',
    'WW': 'Woolworths',
    'Old': 'Old El Paso',
    'Burger': 'Burger Rings',
    'French': 'French Fries'
}

In [ ]:
# Replace inconsistent brand names
transactions["BRAND"] = transactions["BRAND"].replace(brand_mapping)
# Verify standardized brand names
transactions["BRAND"].value_counts()

In [ ]:
# Verify standardized brand names
transactions.head(10)

### Feature Engineering Summary

The following features were successfully created:

- **PACK_SIZE** was extracted from the product name.
- **BRAND** was extracted from the product description.
- Brand names were standardized to eliminate inconsistencies.

These engineered features will be used throughout the exploratory data analysis.

## Feature Validation

Before beginning customer analysis, the newly engineered features are validated to ensure they were extracted correctly and are suitable for further analysis.

## Descriptive Statistics

Descriptive statistics provide an overview of the numerical variables in the transaction dataset, including measures of central tendency, dispersion, and data distribution.

In [ ]:
# Summary statistics for numerical variables
transactions.describe()

In [ ]:
# Investigate unusually large quantity transactions

transactions.sort_values(
    by="PROD_QTY",
    ascending=False
).head(10)

In [ ]:
# Check whether the 200-pack purchases belong to the same customer

transactions[
    transactions["PROD_QTY"] == 200
]

## Observations
- The dataset contains 264,835 transactions after removing one duplicate record.
- Customers purchased an average of 1.91 packs per transaction.
- The average transaction value was $7.30.
    
- The median pack size is 170g, while the middle 50% of transactions fall between 150g and 175g, indicating that these pack sizes are the most common in the dataset.
- The maximum purchase quantity (200 packs) and transaction value ($650) are substantially higher than typical transactions, suggesting the presence of unusually large purchases that are investigated in the following section.

## Categorical Variables

The categorical variables are summarized to understand the number of unique values and their frequency within the dataset.

In [ ]:
# Summary statistics for categorical variables
transactions.describe(include="object")

### Observation

The categorical summary shows that the dataset contains **114 unique product names** across **21 standardized brands**.

**Key observations:**

- The original dataset contained **29 brand names**, including spelling variations and abbreviations.
- After standardizing the brand names, the number of unique brands was reduced to **21**, ensuring consistent brand-level analysis.
- **Kettle** is the most frequently occurring brand, appearing in **41,288 transactions**.
- The most frequently purchased product is **"Kettle Mozzarella Basil & Pesto 175g"**, with **3,304 transactions**.

### Brand Distribution

The frequency of each brand is examined to verify that the brand extraction and standardization steps were successful.

In [ ]:
transactions["BRAND"].value_counts()

## Observations

The brand distribution confirms that the brand standardization process was successful, resulting in 21 consistent brand names.

Key observations:

- Kettle is the most frequently purchased brand, with 41,288 transactions.
- Smiths, Doritos, and Pringles are the next most purchased brands, indicating strong customer preference for these brands.
- Brands such as Burger Rings and French Fries have the fewest transactions, suggesting lower customer demand compared with leading brands.

### Pack Size Distribution

The extracted pack sizes are reviewed to ensure that the feature engineering process correctly identified product sizes.

In [ ]:
transactions["PACK_SIZE"].value_counts().sort_index()

### Observation

The pack size distribution confirms that the feature engineering process successfully extracted product sizes from the product descriptions.

**Key observations:**

- Pack sizes range from **70g to 380g**, indicating a wide variety of product offerings.
- The **175g** pack is the most common, with **66,389 transactions**, followed by **150g (43,131)** and **134g (25,102)**.
- Most purchases are concentrated between **150g and 175g**, suggesting these are the preferred pack sizes among customers.
- Larger pack sizes such as 300g, 330g, and 380g occur less frequently than the most popular pack sizes (150g and 175g), but they still account for a meaningful proportion of transactions.

## Data Integration

The transaction and customer datasets are merged using the **LYLTY_CARD_NBR** field to create a unified dataset for customer-level analysis.

In [ ]:
# Merge transaction and customer datasets
merged_df = pd.merge(
    transactions,
    customers,
    on="LYLTY_CARD_NBR",
    how="left"
)

# Display merged dataset
merged_df.head()


In [ ]:
# Check for missing customer information after merging
merged_df[["LIFESTAGE", "PREMIUM_CUSTOMER"]].isnull().sum()

## Observation
The transaction and customer datasets were successfully merged using the `LYLTY_CARD_NBR` field.

A validation check confirmed that no missing values were introduced in the `LIFESTAGE` and `PREMIUM_CUSTOMER` columns after the merge, indicating that every transaction was successfully matched with a customer record.

The merged dataset combines purchasing behaviour with customer demographic information, providing a complete foundation for customer segmentation and sales analysis.

# Exploratory Data Analysis (EDA)

The purpose of Exploratory Data Analysis (EDA) is to examine customer purchasing behaviour, sales performance, and product preferences. This analysis aims to identify meaningful business insights that support data-driven decision-making and strategic planning.

## Overall Category Performance

This section provides a high-level overview of the chips category by summarizing the overall sales, customer base, and transaction activity during the analysis period.

### Total Category Sales

The total revenue generated from chip sales is calculated to measure the overall performance of the category.

In [ ]:
# Calculate total sales
total_sales = merged_df["TOT_SALES"].sum()

print(f"Total Category Sales: ${total_sales:,.2f}")

### Total Customers

The number of unique loyalty card holders is calculated to determine the size of the customer base.

In [ ]:
# Calculate unique customers
total_customers = merged_df["LYLTY_CARD_NBR"].nunique()

print(f"Total Customers: {total_customers:,}")

### Total Transactions

The total number of transactions is calculated to understand the purchasing activity across all stores.

In [ ]:
# Calculate total transactions
total_transactions = len(merged_df)

print(f"Total Transactions: {total_transactions:,}")

### Observation
-The chips category generated $1.93 million in total sales during the analysis period, providing an overall measure of category revenue.

-A total of 72,637 unique customers made purchases during the analysis period.

-A total of 264,835 transactions were recorded, providing a substantial dataset for analysing customer purchasing behaviour.

-These metrics establish a baseline for evaluating customer segments, spending patterns, and product performance in the subsequent analyses.

## Customer Demographics

Customer demographics are analyzed to understand the composition of the customer base. This section examines customer distribution across different **lifestages** and **premium customer segments**, providing insights into the primary customer groups.

### Customer Distribution by Lifestage

The number of unique customers in each lifestage is calculated to identify the largest customer groups.

In [ ]:
# Number of unique customers by lifestage
lifestage_counts = (
    merged_df.groupby("LIFESTAGE")["LYLTY_CARD_NBR"]
    .nunique()
    .sort_values(ascending=False)
)

print(lifestage_counts)

# Visualize customer distribution by lifestage
plt.figure(figsize=(10,6))

ax = sns.barplot(
    x=lifestage_counts.values,
    y=lifestage_counts.index,
    hue=lifestage_counts.index,
    palette="Blues_r",
    legend=False
)
# Add value labels
for i, value in enumerate(lifestage_counts.values):
    ax.text(
        value + 100,          # Slightly to the right of each bar
        i,                    # Bar position
        f"{value:,}",         # Format with commas
        va="center",
        fontsize=10
    )

plt.title("Customer Distribution by Lifestage", fontsize=14, fontweight="bold")
plt.xlabel("Number of Customers")
plt.ylabel("Lifestage")

plt.tight_layout()
plt.show()

### Observation

The customer base is distributed across seven distinct lifestage segments, with noticeable variation in the number of customers across groups.

**Key observations:**

- **Retirees** represent the largest customer segment with **14,805** unique customers.
- **Older Singles/Couples** (**14,609**) and **Young Singles/Couples** (**14,441**) have customer counts that are very similar to the retiree segment, indicating that these three groups make up a substantial proportion of the customer base.
- **Older Families** and **Young Families** represent medium-sized customer groups.
- **New Families** form the smallest segment, with only **2,549** unique customers.

Overall, the customer base is well distributed across multiple lifestage segments, with retirees and single/couple households representing the largest customer groups.

In [ ]:
# Number of unique customers by premium customer segment
premium_counts = (
    merged_df.groupby("PREMIUM_CUSTOMER")["LYLTY_CARD_NBR"]
    .nunique()
    .sort_values(ascending=False)
)

print(premium_counts)
plt.figure(figsize=(8,5))

ax = sns.barplot(
    x=premium_counts.index,
    y=premium_counts.values,
    hue=premium_counts.index,
    palette="Greens",
    legend=False
)

# Add value labels
for i, value in enumerate(premium_counts.values):
    ax.text(
        i,
        value + 150,
        f"{value:,}",
        ha="center",
        fontsize=10
    )

plt.title("Customer Distribution by Premium Customer Segment",
          fontsize=14,
          fontweight="bold")

plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

### Observation

The customer base is distributed across three premium customer segments, with **Mainstream customers** representing the largest group.

**Key observations:**

- **Mainstream** customers account for the largest customer segment with **29,245** unique customers.
- **Budget** customers represent the second-largest segment with **24,470** customers.
- **Premium** customers form the smallest segment with **18,922** customers.
- The results indicate that the business primarily serves **Mainstream customers**, while Budget and Premium customers also contribute a substantial share of the customer base.

## Sales Performance by Customer Lifestage

Analyzing total sales across customer lifestages helps identify which customer groups contribute the most revenue to the chips category.

In [ ]:
# Total sales by lifestage
sales_lifestage = (
    merged_df.groupby("LIFESTAGE")["TOT_SALES"]
    .sum()
    .sort_values(ascending=False)
)

print(sales_lifestage)

plt.figure(figsize=(10,6))

ax = sns.barplot(
    x=sales_lifestage.values,
    y=sales_lifestage.index,
    hue=sales_lifestage.index,
    palette="Oranges_r",
    legend=False
)

# Add value labels
for i, value in enumerate(sales_lifestage.values):
    ax.text(
        value + 2000,
        i,
        f"${value:,.0f}",
        va="center",
        fontsize=10
    )

plt.title("Total Sales by Customer Lifestage",
          fontsize=14,
          fontweight="bold")

plt.xlabel("Total Sales ($)")
plt.ylabel("Lifestage")

plt.tight_layout()
plt.show()

### Observation


Customer spending varies considerably across different lifestage segments.

**Key observations:**

- **Older Singles/Couples** generated the highest total sales, contributing **$402,420.75**.
- **Retirees** ranked second with **$366,470.90**, followed by **Older Families** with **$353,767.20**.
- **Young Families** also contributed substantially, generating **$316,160.10** in total sales.
- **New Families** recorded the lowest total sales (**$50,433.45**), reflecting their relatively small customer base.
- The results indicate that **Older Singles/Couples, Retirees, and Older Families** contributed the highest total sales during the analysis period. Further analysis is required to determine whether this is driven by a larger customer base, higher spending per customer, or more frequent purchases.

## Sales Performance by Premium Customer Segment

Total sales are analyzed across the three premium customer segments to determine which customer group contributes the highest revenue.

In [ ]:
sales_premium = (
    merged_df.groupby("PREMIUM_CUSTOMER")["TOT_SALES"]
    .sum()
    .sort_values(ascending=False)
)

print(sales_premium)

plt.figure(figsize=(8,5))

ax = sns.barplot(
    x=sales_premium.index,
    y=sales_premium.values,
    hue=sales_premium.index,
    palette="Purples",
    legend=False
)

# Add value labels
for i, value in enumerate(sales_premium.values):
    ax.text(
        i,
        value + 5000,
        f"${value:,.0f}",
        ha="center",
        fontsize=10
    )

plt.title("Total Sales by Premium Customer Segment",
          fontsize=14,
          fontweight="bold")

plt.xlabel("Customer Segment")
plt.ylabel("Total Sales ($)")

plt.tight_layout()
plt.show()

### Observation

- The total sales generated by each premium customer segment reveal differences in revenue contribution.

Key observations:

- Mainstream customers generated the highest total sales, contributing $750,744.50.
    
- Budget customers ranked second with $676,211.55 in total sales.
    
- Premium customers generated the lowest total sales, contributing $507,452.95.
    
- The results indicate that Mainstream customers contributed the largest share of total revenue during the analysis period. Further analysis is required to determine whether this is driven by a larger customer base, higher spending per customer, or more frequent purchases.

## Customer Segment Analysis

Customer segments are analyzed by combining **Lifestage** and **Premium Customer** classifications. This analysis helps identify which customer groups contribute the highest sales and provides valuable insights for targeted marketing strategies.

In [ ]:
# Total sales by customer segment
segment_sales = (
    merged_df.groupby(["LIFESTAGE", "PREMIUM_CUSTOMER"])["TOT_SALES"]
    .sum()
    .unstack()
)

print(segment_sales)
segment_sales = (
    merged_df.groupby(["LIFESTAGE", "PREMIUM_CUSTOMER"])["TOT_SALES"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(12,7))

ax = sns.barplot(
    data=segment_sales,
    y="LIFESTAGE",
    x="TOT_SALES",
    hue="PREMIUM_CUSTOMER",
    palette="Set2"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=8)

plt.title("Total Sales by Customer Segment", fontsize=15, fontweight="bold")
plt.xlabel("Total Sales ($)")
plt.ylabel("Lifestage")

plt.tight_layout()
plt.show()

### Observation

The combined customer segment analysis reveals substantial differences in sales performance across customer groups.

**Key observations:**

- **Budget Older Families** generated the highest total sales among all customer segments, contributing **$168,363.25**.
- **Mainstream Young Singles/Couples** (**$157,621.60**) and **Mainstream Retirees** (**$155,677.05**) also contributed substantially to total sales.
- **Older Singles/Couples** consistently generated high sales across all three premium customer segments, indicating strong revenue contribution regardless of customer type.
- **New Families** recorded the lowest total sales across all premium customer segments, reflecting their relatively small customer base.
- Overall, the results identify **Budget Older Families, Mainstream Young Singles/Couples, Mainstream Retirees, and Older Singles/Couples** as the largest contributors to total sales. Further analysis is required to determine whether these results are driven by customer numbers, purchase frequency, or average spending.

## Average Spend per Transaction

Total sales alone do not indicate how much customers spend during each shopping trip. Therefore, the average spend per transaction is calculated for each customer segment to identify customers with higher purchasing value.

In [ ]:
# Average spend per transaction by customer segment
avg_spend = (
    merged_df.groupby(["LIFESTAGE", "PREMIUM_CUSTOMER"])["TOT_SALES"]
    .mean()
    .reset_index()
)

print(avg_spend)
plt.figure(figsize=(12,7))

ax = sns.barplot(
    data=avg_spend,
    x="LIFESTAGE",
    y="TOT_SALES",
    hue="PREMIUM_CUSTOMER",
    palette="Set2"
)

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt="$%.2f", fontsize=8)

plt.title("Average Spend per Transaction by Customer Segment",
          fontsize=15,
          fontweight="bold")

plt.xlabel("Lifestage")
plt.ylabel("Average Spend ($)")
plt.xticks(rotation=25)

plt.legend(title="Premium Customer")

plt.tight_layout()
plt.show()

### Observation

The average spend per transaction is relatively consistent across most customer segments, with only small variations.

**Key observations:**

- **Retirees (Premium)** recorded the highest average spend per transaction at **$7.46**, closely followed by **Older Singles/Couples (Premium)** at **$7.45**.
- **Budget Retirees** and **Budget Older Singles/Couples** also spent more than the overall average per transaction.
- **Young Singles/Couples (Budget)** recorded the lowest average spend per transaction at **$6.62**, followed closely by **Young Singles/Couples (Premium)** at **$6.63**.
- The relatively small differences in average transaction value suggest that variations in total sales are more likely driven by customer volume and transaction frequency than by substantially higher spending per shopping trip.


## Average Quantity Purchased per Transaction

The average number of chip packs purchased per transaction is analyzed across customer segments. This helps determine whether higher sales are driven by customers purchasing larger quantities or by purchasing higher-priced products.

In [ ]:
# Average quantity purchased per transaction
avg_qty = (
    merged_df.groupby(["LIFESTAGE", "PREMIUM_CUSTOMER"])["PROD_QTY"]
    .mean()
    .reset_index()
)

print(avg_qty)
plt.figure(figsize=(12,7))

ax = sns.barplot(
    data=avg_qty,
    x="LIFESTAGE",
    y="PROD_QTY",
    hue="PREMIUM_CUSTOMER",
    palette="Set2"
)

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=8)

plt.title("Average Quantity Purchased per Transaction by Customer Segment",
          fontsize=15,
          fontweight="bold")

plt.xlabel("Lifestage")
plt.ylabel("Average Quantity Purchased")
plt.xticks(rotation=25)

plt.legend(title="Premium Customer")

plt.tight_layout()
plt.show()

### Observation

The average quantity purchased per transaction is relatively consistent across customer segments, with only slight differences.

**Key observations:**

- **Older Families (Premium)** purchased the highest average quantity, buying **1.98 packs** per transaction.
- **Older Families** and **Young Families** consistently purchased around **1.94–1.98 packs** per transaction, indicating a tendency to buy slightly larger quantities.
- **Young Singles/Couples (Budget)** purchased the lowest average quantity, averaging **1.80 packs** per transaction.
- Overall, most customer segments purchased **between 1.8 and 2.0 packs per transaction**, suggesting that differences in total sales are unlikely to be explained by substantially larger basket sizes alone.

# Brand Performance

Brand performance is analyzed to identify the highest-selling chip brands and understand customer preferences. This analysis helps determine which brands contribute the most to overall category sales.

## Total Sales by Brand

Total sales are calculated for each brand to identify the strongest-performing brands in the chips category.

In [ ]:
# Total sales by brand
brand_sales = (
    merged_df.groupby("BRAND")["TOT_SALES"]
    .sum()
    .sort_values(ascending=False)
)

print(brand_sales)
plt.figure(figsize=(12,7))

ax = sns.barplot(
    x=brand_sales.values,
    y=brand_sales.index,
    hue=...,
    palette="viridis",
    legend=False
)

# Add value labels
for i, value in enumerate(brand_sales.values):
    ax.text(
        value + 2000,
        i,
        f"${value:,.0f}",
        va="center",
        fontsize=9
    )

plt.title("Total Sales by Brand", fontsize=15, fontweight="bold")
plt.xlabel("Total Sales ($)")
plt.ylabel("Brand")

plt.tight_layout()
plt.show()

### Observation

Brand performance analysis shows that sales are concentrated among a small number of leading chip brands.

**Key observations:**

- **Kettle** is the highest-performing brand, generating **$390,239.80** in total sales, making it the strongest contributor to category revenue.
- **Doritos** and **Smiths** are the next best-performing brands, generating **$241,890.90** and **$224,654.20** respectively.
- **Pringles** also demonstrates strong performance with **$177,655.50** in sales.
- Mid-performing brands such as **Infuzions, Red Rock Deli, Old El Paso, Thins, and Twisties** contribute meaningful revenue to the category.
- Lower-performing brands including **Burger Rings, French Fries, and Sunbites** contribute relatively smaller amounts compared with the leading brands.
- Overall, the results indicate that a small group of major brands drives a significant proportion of chips category sales, suggesting that these brands are important for maintaining category performance.

## Brand Sales Contribution

The percentage contribution of each brand to total category sales is calculated to identify which brands have the greatest impact on overall revenue performance.

In [ ]:
brand_contribution = (
    brand_sales / brand_sales.sum() * 100
).round(2)

print(brand_contribution.head(10))

plt.figure(figsize=(10,6))

top10_contribution = brand_contribution.head(10)

ax = sns.barplot(
    x=top10_contribution.values,
    y=top10_contribution.index,
    hue=top10_contribution.index,
    palette="viridis",
    legend=False
)

for i, value in enumerate(top10_contribution.values):
    ax.text(
        value + 0.2,
        i,
        f"{value:.2f}%",
        va="center",
        fontsize=10
    )

plt.title("Top 10 Brands Contribution to Total Sales",
          fontsize=15,
          fontweight="bold")

plt.xlabel("Sales Contribution (%)")
plt.ylabel("Brand")

plt.tight_layout()
plt.show()

### Observation

Brand contribution analysis shows that category revenue is concentrated among a small number of leading brands.

**Key observations:**

- **Kettle** contributes the highest share of category sales, accounting for **20.17%** of total revenue.
- **Doritos** and **Smiths** are the next largest contributors, representing **12.50%** and **11.61%** of total sales respectively.
- The top four brands (**Kettle, Doritos, Smiths, and Pringles**) contribute approximately **53% of total category revenue**.
- Other brands contribute smaller individual shares, indicating that the chips category is driven mainly by a few strong-performing brands.
- These findings suggest that maintaining availability and promotional support for leading brands is important for maximizing category performance.

# Pack Size Performance

Pack size analysis helps identify customer preferences for different product sizes. Understanding which pack sizes generate the highest sales can support inventory planning, pricing strategies, and product assortment decisions.

## Total Sales by Pack Size

Total sales are calculated for each pack size to determine which product sizes contribute the most revenue.

In [ ]:
# Total sales by pack size
pack_sales = (
    merged_df.groupby("PACK_SIZE")["TOT_SALES"]
    .sum()
    .sort_values(ascending=False)
)

print(pack_sales)
plt.figure(figsize=(12,6))

ax = sns.barplot(
    x=pack_sales.index.astype(str),
    y=pack_sales.values,
    hue=pack_sales.index.astype(str),
    palette="viridis",
    legend=False
)

# Add value labels
for i, value in enumerate(pack_sales.values):
    ax.text(
        i,
        value + 2000,
        f"${value:,.0f}",
        ha="center",
        fontsize=8
    )

plt.title("Total Sales by Pack Size", fontsize=15, fontweight="bold")
plt.xlabel("Pack Size (g)")
plt.ylabel("Total Sales ($)")

plt.tight_layout()
plt.show()

### Observation

The pack size analysis shows clear customer preferences for specific product sizes, with medium-sized packs generating the highest revenue.

**Key observations:**

- The **175g pack size** generated the highest sales, contributing **$485,431.40** in revenue.
    
- The **150g pack size** was the second strongest performer, generating **$304,288.50** in sales.
    
- The **134g pack size** ranked third with **$177,655.50** in revenue.
    
- Pack sizes between **150g and 175g** account for a substantial share of category sales and are the most popular pack sizes among customers.
- Larger pack sizes such as **300g, 330g, and 380g** also contribute meaningful revenue, indicating demand for larger pack options.
- Smaller pack sizes below **125g** generate relatively lower sales, suggesting lower customer demand.

Overall, customers show a strong preference for **standard medium-sized chip packs**, particularly **150g and 175g** products. These pack sizes represent the strongest-performing products and should be considered in product assortment and promotional planning.

# Monthly Sales Trend

Monthly sales performance is analyzed to identify purchasing trends over the 12-month analysis period. Understanding sales patterns over time helps identify seasonal behaviour and demand consistency.

In [ ]:
# Extract month from transaction date
merged_df["MONTH"] = merged_df["DATE"].dt.to_period("M").astype(str)

# Calculate monthly sales
monthly_sales = (
    merged_df.groupby("MONTH")["TOT_SALES"]
    .sum()
)

print(monthly_sales)
plt.figure(figsize=(12,5))

plt.plot(
    monthly_sales.index,
    monthly_sales.values,
    marker="o"
)

plt.title("Monthly Sales Trend", fontsize=15, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Total Sales ($)")

plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Observation

Monthly sales remained relatively stable throughout the 12-month analysis period, indicating consistent customer demand for chip products.

**Key observations:**

- **December 2018** recorded the highest monthly sales, generating **$167,913.40**.
                                                                                                                        
- **March 2019** also showed strong performance, with **$166,265.20** in sales.
    
- **February 2019** recorded the lowest monthly sales at **$150,665.00**.
    
- Monthly sales remained within a relatively narrow range of approximately **$150,000 to $168,000**, indicating stable purchasing patterns throughout the year.
- No significant long-term upward or downward sales trend was observed over the analysis period.

## Target Customer Segment

The target customer segment is identified by comparing total sales across customer lifestage and premium customer groups. Identifying the highest-value customer segment helps understand which customers contribute the most revenue and provides a foundation for targeted marketing and business recommendations.

In [ ]:
target_segment = (
    merged_df.groupby(["LIFESTAGE", "PREMIUM_CUSTOMER"])["TOT_SALES"]
    .sum()
    .sort_values(ascending=False)
)

print(target_segment)

### Observation

The customer segment analysis identifies **Budget Older Families** as the highest revenue-generating customer segment, contributing **$168,363.25** in total sales.

**Key observations:**

- **Budget Older Families** generated the highest total sales among all customer segments.
    
- **Mainstream Young Singles/Couples** ranked second with **$157,621.60** in total sales.
    
- **Mainstream Retirees** ranked third, contributing **$155,677.05**.
    
- The results indicate that **Budget Older Families** are the most valuable customer segment based on total revenue.

The following analyses examine the brand and pack size preferences of **Budget Older Families** to better understand their purchasing behaviour and support targeted business recommendations.

## Brand Preference of the Target Customer

The purchasing preferences of the target customer segment (Budget Older Families) are analyzed to identify the brands that contribute the most to their purchases. This helps determine which brands are most popular among the highest-value customer segment.

In [ ]:
# Filter the target customer segment
target_customers = merged_df[
    (merged_df["LIFESTAGE"] == "OLDER FAMILIES") &
    (merged_df["PREMIUM_CUSTOMER"] == "Budget")
]

# Total sales by brand
target_brand_sales = (
    target_customers.groupby("BRAND")["TOT_SALES"]
    .sum()
    .sort_values(ascending=False)
)

print(target_brand_sales)

### Observation

The brand preference analysis identifies the brands most frequently purchased by the highest-value customer segment, **Budget Older Families**.

**Key observations:**

- **Kettle** is the most preferred brand, generating **$32,058.00** in sales.
    
- **Smiths** ranks second with **$21,651.10**, followed by **Doritos** with **$20,164.95**.
    
- **Pringles** is the fourth highest-selling brand, contributing **$14,300.50**.
    
- Brands such as **Red Rock Deli, Infuzions, Old El Paso, and Thins** also generate meaningful sales within this customer segment.
- Lower-selling brands, including **Burger Rings, French Fries, and Sunbites**, contribute relatively little to total sales.

Overall, **Budget Older Families** show a strong preference for **Kettle, Smiths, and Doritos**, making these brands the strongest contributors to sales within the highest-value customer segment.

## Pack Size Preference of the Target Customer

The preferred pack sizes of the target customer segment (Budget Older Families) are analyzed to identify the product sizes that contribute the most to sales. This helps understand purchasing preferences and supports product assortment decisions.

In [ ]:
# Total sales by pack size for Budget Older Families
target_pack_sales = (
    target_customers.groupby("PACK_SIZE")["TOT_SALES"]
    .sum()
    .sort_values(ascending=False)
)

print(target_pack_sales)

plt.figure(figsize=(12,6))

ax = sns.barplot(
    x=target_pack_sales.index.astype(str),
    y=target_pack_sales.values,
    hue=target_pack_sales.index.astype(str),
    palette="viridis",
    legend=False
)

# Add value labels
for i, value in enumerate(target_pack_sales.values):
    ax.text(
        i,
        value + 150,
        f"${value:,.0f}",
        ha="center",
        fontsize=8
    )

plt.title("Pack Size Preference of Budget Older Families",
          fontsize=15,
          fontweight="bold")

plt.xlabel("Pack Size (g)")
plt.ylabel("Total Sales ($)")

plt.tight_layout()
plt.show()

### Observation

The pack size preference analysis shows that **Budget Older Families** have purchasing patterns similar to the overall customer base, with medium-sized packs generating the highest sales.

**Key observations:**

- The **175g pack size** generated the highest sales, contributing **$42,204.70**.
    
- The **150g pack size** ranked second with ** $27,017.10 **, followed by the ** 134g pack size** with **$14,300.50**.
    
    
- Pack sizes **110g**, **170g**, and **330g** also contributed substantially to total sales.
- Smaller pack sizes such as **70g**, **90g**, and **125g** generated comparatively lower sales.
- Overall, **Budget Older Families** show a clear preference for **medium-sized pack sizes (150g–175g)**, consistent with the overall purchasing patterns observed across the dataset.

## Business Recommendations

The analysis identified the highest-value customer segment, their preferred brands, and preferred pack sizes. Based on these findings, the following recommendations are proposed to improve sales performance and support data-driven business decisions.

### Recommendations

Based on the analysis, the following recommendations are proposed:

1. **Prioritize Budget Older Families**
   - Budget Older Families generated the highest total sales and represent the most valuable customer segment.
   - Marketing campaigns and promotional activities should be designed to better engage this customer group.

2. **Maintain Availability of Leading Brands**
   - Kettle, Smiths, and Doritos are the most popular brands among the target customer segment.
   - Ensuring consistent availability of these brands can help maintain strong sales performance.

3. **Focus on Medium-Sized Pack Options**
   - Pack sizes between 150g and 175g generated the highest sales across both the overall customer base and the target customer segment.
   - These pack sizes should receive priority in product assortment and promotional activities.

4. **Support High-Performing Products**
   - The strongest-selling brand and pack size combinations should be highlighted in promotions and in-store displays to maximize customer engagement.

5. **Continue Monitoring Customer Behaviour**
   - Customer preferences may change over time.
   - Regular analysis of sales and purchasing behaviour can help identify emerging trends and support future business decisions.

## Conclusion

This project analyzed customer purchasing behaviour in the chips category by combining transaction and customer data. The analysis examined sales performance, customer demographics, product preferences, and customer segments to identify key business insights and support data-driven decision-making.

### Conclusion

This analysis provided a comprehensive overview of customer purchasing behaviour within the chips category.

**Key findings include:**

- The chips category generated approximately **$1.93 million** in sales from **264,835 transactions** completed by **72,637 unique customers**.
- **Budget Older Families** were identified as the highest-value customer segment based on total sales.
- **Kettle** was the highest-performing brand, followed by **Smiths** and **Doritos**.
- **Medium-sized pack sizes (150g–175g)** generated the highest sales across both the overall customer base and the target customer segment.
- Monthly sales remained relatively stable throughout the analysis period, indicating consistent customer demand.

Overall, the findings provide practical insights into customer behaviour and product performance. These insights can support marketing strategies, product assortment decisions, and future business planning aimed at improving category performance.